In [11]:
import math

In [12]:
# ==========================================
# STEP 1: Input sentence
# ==========================================

text = "what is the main purpose of transformer architecture"
tokens = text.split()

print("Tokens:", tokens)

Tokens: ['what', 'is', 'the', 'main', 'purpose', 'of', 'transformer', 'architecture']


In [13]:
# ==========================================
# STEP 2: Create simple input vectors
# ==========================================

# In a real Transformer, these vectors come
# from embedding + positional information.

X = []

for i, token in enumerate(tokens):
    X.append([
        (i + 1) * 0.1,
        (i + 1) * 0.2,
        (i + 1) * 0.3,
        (i + 1) * 0.4
    ])

In [14]:
# ==========================================
# STEP 3: Define matrix multiplication
# ==========================================

def transpose(matrix):
    return [list(row) for row in zip(*matrix)]


def matmul(A, B):
    result = []

    for row in A:
        new_row = []

        for col in transpose(B):
            value = sum(
                row[i] * col[i]
                for i in range(len(col))
            )
            new_row.append(value)

        result.append(new_row)

    return result

In [15]:
# ==========================================
# STEP 4: Create Q, K, V
# ==========================================

# Identity matrices make Q = K = V = X
# for this simplified demonstration.
# Real models learn separate projection matrices.

Q = [row[:] for row in X]
K = [row[:] for row in X]
V = [row[:] for row in X]

print("\nQuery vectors:", Q)
print("\nKey vectors:", K)
print("\nValue vectors:", V)



Query vectors: [[0.1, 0.2, 0.3, 0.4], [0.2, 0.4, 0.6, 0.8], [0.30000000000000004, 0.6000000000000001, 0.8999999999999999, 1.2000000000000002], [0.4, 0.8, 1.2, 1.6], [0.5, 1.0, 1.5, 2.0], [0.6000000000000001, 1.2000000000000002, 1.7999999999999998, 2.4000000000000004], [0.7000000000000001, 1.4000000000000001, 2.1, 2.8000000000000003], [0.8, 1.6, 2.4, 3.2]]

Key vectors: [[0.1, 0.2, 0.3, 0.4], [0.2, 0.4, 0.6, 0.8], [0.30000000000000004, 0.6000000000000001, 0.8999999999999999, 1.2000000000000002], [0.4, 0.8, 1.2, 1.6], [0.5, 1.0, 1.5, 2.0], [0.6000000000000001, 1.2000000000000002, 1.7999999999999998, 2.4000000000000004], [0.7000000000000001, 1.4000000000000001, 2.1, 2.8000000000000003], [0.8, 1.6, 2.4, 3.2]]

Value vectors: [[0.1, 0.2, 0.3, 0.4], [0.2, 0.4, 0.6, 0.8], [0.30000000000000004, 0.6000000000000001, 0.8999999999999999, 1.2000000000000002], [0.4, 0.8, 1.2, 1.6], [0.5, 1.0, 1.5, 2.0], [0.6000000000000001, 1.2000000000000002, 1.7999999999999998, 2.4000000000000004], [0.70000000000

In [16]:
# ==========================================
# STEP 5: Calculate attention scores
# ==========================================

K_transposed = transpose(K)

scores = matmul(Q, K_transposed)

# Scale scores by sqrt(key dimension)
d_k = len(K[0])
scale = math.sqrt(d_k)

scaled_scores = [
    [value / scale for value in row]
    for row in scores
]

print("\nScaled attention scores:")

for token, row in zip(tokens, scaled_scores):
    print(token, [round(x, 3) for x in row])


Scaled attention scores:
what [0.15, 0.3, 0.45, 0.6, 0.75, 0.9, 1.05, 1.2]
is [0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.4]
the [0.45, 0.9, 1.35, 1.8, 2.25, 2.7, 3.15, 3.6]
main [0.6, 1.2, 1.8, 2.4, 3.0, 3.6, 4.2, 4.8]
purpose [0.75, 1.5, 2.25, 3.0, 3.75, 4.5, 5.25, 6.0]
of [0.9, 1.8, 2.7, 3.6, 4.5, 5.4, 6.3, 7.2]
transformer [1.05, 2.1, 3.15, 4.2, 5.25, 6.3, 7.35, 8.4]
architecture [1.2, 2.4, 3.6, 4.8, 6.0, 7.2, 8.4, 9.6]


In [17]:
# ==========================================
# STEP 6: Apply softmax
# ==========================================

def softmax(values):
    # Subtract maximum for numerical stability.
    max_value = max(values)

    exps = [
        math.exp(value - max_value)
        for value in values
    ]

    total = sum(exps)

    return [value / total for value in exps]


attention_weights = [
    softmax(row)
    for row in scaled_scores
]

print("\nAttention weights:")

for token, row in zip(tokens, attention_weights):
    print(token, [round(x, 3) for x in row])


Attention weights:
what [0.07, 0.081, 0.094, 0.109, 0.127, 0.148, 0.172, 0.199]
is [0.035, 0.047, 0.064, 0.086, 0.116, 0.156, 0.211, 0.285]
the [0.016, 0.025, 0.039, 0.062, 0.097, 0.151, 0.238, 0.373]
main [0.007, 0.012, 0.023, 0.041, 0.075, 0.137, 0.25, 0.455]
purpose [0.003, 0.006, 0.012, 0.026, 0.056, 0.118, 0.25, 0.529]
of [0.001, 0.003, 0.007, 0.016, 0.04, 0.098, 0.241, 0.594]
transformer [0.0, 0.001, 0.003, 0.01, 0.028, 0.08, 0.228, 0.65]
architecture [0.0, 0.001, 0.002, 0.006, 0.019, 0.063, 0.21, 0.699]


In [18]:
# ==========================================
# STEP 7: Weighted sum of Value vectors
# ==========================================

output = []

for weights in attention_weights:
    output_vector = []

    for dimension in range(len(V[0])):
        value = sum(
            weights[j] * V[j][dimension]
            for j in range(len(V))
        )

        output_vector.append(value)

    output.append(output_vector)

In [19]:
# ==========================================
# STEP 8: Display contextual vectors
# ==========================================

print("\nSelf-attention output:")

for token, vector in zip(tokens, output):
    print(token, [round(x, 4) for x in vector])


Self-attention output:
what [0.5269, 1.0538, 1.5807, 2.1076]
is [0.594, 1.188, 1.782, 2.3759]
the [0.6465, 1.293, 1.9395, 2.5861]
main [0.685, 1.37, 2.055, 2.74]
purpose [0.7125, 1.4249, 2.1374, 2.8498]
of [0.7321, 1.4642, 2.1963, 2.9283]
transformer [0.7463, 1.4927, 2.239, 2.9854]
architecture [0.757, 1.5139, 2.2709, 3.0278]
